# Heater Current Drift Detection
## Exploratory Data Analysis EDA
Author: Sandro Wiedmer

Q2/2026

## Descriptive statistics (scroll down for plots of drift and healthy examples):
- "total dataset: rows x columns",
- "MF-450 dataset: rows x columns",
- "damaged pairs dataset: rows x columns",
- "total dataset: heater current samples",
- "MF-450 dataset: heater current samples",
- "damaged pairs dataset: heater current samples",
- "total dataset: number of device-tube pairs",
- "MF-450 dataset: number of device-tube pairs",
- "damaged pairs dataset: number of device-tube pairs"

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# --- 1. LOAD RAW DATA ---
df_raw = spark.table("svc_datacore.external.silver_blox_enriched")

# --- 2. ISOLATE ALL 450MF COMBOS ---
df_valid_450mf_combos = df_raw.filter(
    F.lower(F.col("tube_model")).like("%450mf%") | 
    F.lower(F.col("tube_model")).like("%y.tu450%")
).select("device_id", "tube_id").distinct()

df_tube = df_raw.join(df_valid_450mf_combos, on=["device_id", "tube_id"], how="inner")

# --- 3. FILTER EVENTS & CLEAN DATA ---
df_tube = df_tube.filter(F.col("event_type").isin("exposure", "warm_up", "disrupted_warm_up"))
selected_features = ["focal_spot", "hv_kv_set", "tucu_ma_set", "pwr_w_reached", "grid_voltage"]
df_clean = df_tube.dropna(subset=["start_time", "filcu_a"] + selected_features)

In [0]:
# manually confirmed "true positives"
DAMAGED_PAIRS = [
  ("t3-1678901-1966","1674634"),
  #("t3-1797055-2150","1787334"),  # MF-225
  ("t3-1831151-186","1824362"),
  ("t3-1882150-500","1864990"),
  ("t3-1991730-874","1977123"),
  ("t3-2008-1726","1892070"),
  ("t3-2008-1726","2043143"),
  ("t3-2009-1777","1448576"),
  ("t3-2002-1557","1392867"),
  ("t3-1817879-52","1810944"),
  ("t3-1993491-843","1950888"),
  ("t3-1882151-611","1853962"),
  ("t3-1829086-180","1939694"),
  ("t3-1830503-165","1823505"),
  
]


In [0]:
# 1. number of rows x columns of the whole data set available (df_raw)
shape_df_raw = (df_raw.count(), len(df_raw.columns))

# 2. number of rows x columns of the dataset when filtered for MF-450 (df_tube)
shape_df_tube = (df_tube.count(), len(df_tube.columns))

# 3. number of rows x columns of the dataset when filtered for damaged pairs (DAMAGED_PAIRS, 7 pcs.)
damaged_pairs_df = df_raw.filter(
    F.struct("device_id", "tube_id").isin([F.struct(F.lit(d), F.lit(t)) for d, t in DAMAGED_PAIRS])
)
shape_damaged_pairs = (damaged_pairs_df.count(), len(damaged_pairs_df.columns))

# 4. number of pairs available in df_raw
num_pairs_df_raw = df_raw.select("device_id", "tube_id").distinct().count()

# 5. number of pairs of MF-450 available (df_tube)
num_pairs_df_tube = df_tube.select("device_id", "tube_id").distinct().count()

# 6. number of heater current samples ("filcu_a") in df_raw, df_tube, and df_tube DAMAGED_PAIRS only
num_filcu_a_df_raw = df_raw.filter(F.col("filcu_a").isNotNull()).count()
num_filcu_a_df_tube = df_tube.filter(F.col("filcu_a").isNotNull()).count()
damaged_pairs_tube_df = df_tube.filter(
    F.struct("device_id", "tube_id").isin([F.struct(F.lit(d), F.lit(t)) for d, t in DAMAGED_PAIRS])
)
num_filcu_a_df_tube_damaged = damaged_pairs_tube_df.filter(F.col("filcu_a").isNotNull()).count()

num_damaged_pairs = len(DAMAGED_PAIRS)

import pandas as pd

stats = pd.DataFrame({
    "description": [
        "total dataset: rows x columns",
        "MF-450 dataset: rows x columns",
        "damaged pairs dataset: rows x columns",
        "total dataset: heater current samples",
        "MF-450 dataset: heater current samples",
        "damaged pairs dataset: heater current samples",
        "total dataset: number of device-tube pairs",
        "MF-450 dataset: number of device-tube pairs",
        "damaged pairs dataset: number of device-tube pairs"
    ],
    "value": [
        f"{shape_df_raw[0]} x {shape_df_raw[1]}",
        f"{shape_df_tube[0]} x {shape_df_tube[1]}",
        f"{shape_damaged_pairs[0]} x {shape_damaged_pairs[1]}",
        num_filcu_a_df_raw,
        num_filcu_a_df_tube,
        num_filcu_a_df_tube_damaged,
        num_pairs_df_raw,
        num_pairs_df_tube,
        num_damaged_pairs
    ]
})

display(stats)


/databricks/spark/python/pyspark/sql/pandas/conversion.py:783: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Expected bytes, got a 'int' object
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


description,value
total dataset: rows x columns,37940853 x 45
MF-450 dataset: rows x columns,290122 x 45
damaged pairs dataset: rows x columns,119758 x 45
total dataset: heater current samples,11840069
MF-450 dataset: heater current samples,289975
damaged pairs dataset: heater current samples,49308
total dataset: number of device-tube pairs,1587
MF-450 dataset: number of device-tube pairs,164
damaged pairs dataset: number of device-tube pairs,13


Histogram of heater current samples (filcu_a) per pair, damaged_pairs and not damaged pairs in a separate color

In [0]:

import plotly.graph_objs as go

# Prepare per-pair counts for damaged and healthy pairs
damaged_pairs_set = set(DAMAGED_PAIRS)

# Count filcu_a samples per pair for damaged pairs
damaged_counts_df = df_tube.filter(
    F.struct("device_id", "tube_id").isin([F.struct(F.lit(d), F.lit(t)) for d, t in DAMAGED_PAIRS])
).filter(F.col("filcu_a").isNotNull()) \
 .groupBy("device_id", "tube_id") \
 .agg(F.count("filcu_a").alias("num_samples")) \
 .withColumn("pair_type", F.lit("damaged"))

# Count filcu_a samples per pair for healthy pairs
healthy_pairs_df = df_tube.select("device_id", "tube_id").distinct() \
    .filter(~F.struct("device_id", "tube_id").isin([F.struct(F.lit(d), F.lit(t)) for d, t in DAMAGED_PAIRS]))

healthy_counts_df = df_tube.join(healthy_pairs_df, on=["device_id", "tube_id"], how="inner") \
    .filter(F.col("filcu_a").isNotNull()) \
    .groupBy("device_id", "tube_id") \
    .agg(F.count("filcu_a").alias("num_samples")) \
    .withColumn("pair_type", F.lit("healthy"))

# Combine both
all_counts_df = healthy_counts_df.unionByName(damaged_counts_df)
pdf = all_counts_df.toPandas()

# Display statistics for number of heater current samples per pair
import numpy as np
import pandas as pd

def stats_for_group(data):
    return pd.Series({
        "min": int(np.min(data)) if len(data) else None,
        "max": int(np.max(data)) if len(data) else None,
        "mean": float(np.mean(data)) if len(data) else None,
        "median": float(np.median(data)) if len(data) else None,
        "std": float(np.std(data, ddof=1)) if len(data) > 1 else None
    })

stats_total = stats_for_group(pdf["num_samples"])
stats_damaged = stats_for_group(pdf[pdf["pair_type"] == "damaged"]["num_samples"])
stats_healthy = stats_for_group(pdf[pdf["pair_type"] == "healthy"]["num_samples"])

stats_df = pd.DataFrame({
    "total": stats_total,
    "damaged": stats_damaged,
    "healthy": stats_healthy
}).reset_index().rename(columns={"index": "statistic"})

display(stats_df)

# Plot histogram: healthy first, then damaged, with finer x-axis bins
fig = go.Figure()
num_bins = 100
bin_size = max(1, int((pdf["num_samples"].max() - pdf["num_samples"].min()) / num_bins))
for pair_type, color in [("healthy", "blue"), ("damaged", "red")]:
    data = pdf[pdf["pair_type"] == pair_type]["num_samples"]
    fig.add_trace(go.Histogram(
        x=data,
        name=pair_type,
        marker_color=color,
        opacity=0.7,
        xbins=dict(
            start=pdf["num_samples"].min(),
            end=pdf["num_samples"].max(),
            size=bin_size
        )
    ))
fig.update_layout(
    barmode='overlay',
    title="Histogram of heater current samples per pair",
    xaxis_title="Number of heater current samples per pair",
    yaxis_title="Count of pairs",
    legend_title="Pair Type", 
    # xaxis_type="log",  # Removed log scale for x-axis
    yaxis_type="log"   # Keep/remove as needed
)
display(fig)


statistic,total,damaged,healthy
min,1.0,478.0,1.0
max,30008.0,16820.0,30008.0
mean,1768.1402439024391,3792.923076923077,1593.8211920529802
median,232.0,2020.0,170.0
std,4336.4595676009685,4604.458825630117,4283.999732471347


## Plots of drift examples: Manually confirmed true positives **TP**
## 

In [0]:
import plotly.express as px

for device_id, tube_id in DAMAGED_PAIRS:
    pair_df = df_raw.filter(
        (F.col("device_id") == device_id) & (F.col("tube_id") == tube_id)
    ).select(
        "start_time", "filcu_a", "focal_spot", "hv_kv_set", "tucu_ma_set"
    ).orderBy("start_time")
    # Create a combined label for coloring
    pair_df = pair_df.withColumn(
        "operating_point",
        F.concat_ws(
            " | ",
            F.concat(F.lit("Focal Spot: "), F.col("focal_spot")),
            F.concat(F.lit("High Voltage (kV): "), F.col("hv_kv_set")),
            #F.concat(F.lit("Emission Current (mA): "), F.col("tucu_ma_set"))
        )
    )
    pdf = pair_df.toPandas()
    if pdf.empty:
        continue
    op_counts = pdf["operating_point"].value_counts()
    ordered_ops = op_counts.index.tolist()
    color_sequence = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24 + px.colors.qualitative.Light24
    marker_symbols = ['circle', 'x', 'star', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'triangle-left', 'triangle-right', 'square', 'pentagon', 'hexagon', 'hexagram', 'star-triangle-up', 'star-triangle-down', 'star-square', 'star-diamond', 'hourglass', 'bowtie']
    op_styles = {}
    for i, op in enumerate(ordered_ops):
        color = color_sequence[i % len(color_sequence)]
        symbol = marker_symbols[i % len(marker_symbols)]
        op_styles[op] = (color, symbol)
    import plotly.graph_objs as go
    fig = go.Figure()
    for op in ordered_ops:
        color, symbol = op_styles[op]
        op_data = pdf[pdf["operating_point"] == op]
        fig.add_trace(go.Scatter(
            x=op_data["start_time"],
            y=op_data["filcu_a"],
            mode='markers',
            name=op,
            marker=dict(color=color, symbol=symbol, size=8),
            showlegend=True
        ))
    fig.update_layout(
        title=f"Device SN: {device_id}, Tube SN: {tube_id}",
        xaxis_title="Timestamp",
        yaxis_title="Heater Current (A)",
        legend_title="Operating Point"
    )
    display(fig)

Plot healthy resp. not damaged pairs (defined number of examples)

In [0]:

import plotly.express as px

def plot_healthy_pairs(num_pairs=3, random_seed=13):
    # Get all unique (device_id, tube_id) pairs in df_tube that are NOT in DAMAGED_PAIRS
    damaged_set = set(DAMAGED_PAIRS)
    healthy_pairs_df = (
        df_tube.select("device_id", "tube_id")
        .distinct()
        .filter(~F.concat_ws('_', F.col("device_id"), F.col("tube_id")).isin([f'{pair[0]}_{pair[1]}' for pair in DAMAGED_PAIRS])))
    
    # Sample num_pairs randomly
    sampled_pairs = healthy_pairs_df.orderBy(F.rand(seed=random_seed)).limit(num_pairs)
    healthy_pairs = [(row["device_id"], row["tube_id"]) for row in sampled_pairs.collect()]

    for device_id, tube_id in healthy_pairs:
        pair_df = df_tube.filter(
            (F.col("device_id") == device_id) & (F.col("tube_id") == tube_id)
        ).select(
            "start_time", "filcu_a", "focal_spot", "hv_kv_set", "tucu_ma_set"
        ).orderBy("start_time")
        pair_df = pair_df.withColumn(
            "operating_point",
            F.concat_ws(
                " | ",
                F.concat(F.lit("Focal Spot: "), F.col("focal_spot")),
                F.concat(F.lit("High Voltage (kV): "), F.col("hv_kv_set")),
                #F.concat(F.lit("Emission Current (mA): "), F.col("tucu_ma_set"))
            )
        )
        pdf = pair_df.toPandas()
        if pdf.empty:
            continue
        op_counts = pdf["operating_point"].value_counts()
        ordered_ops = op_counts.index.tolist()
        color_sequence = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24 + px.colors.qualitative.Light24
        marker_symbols = ['circle', 'x', 'star', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'triangle-left', 'triangle-right', 'square', 'pentagon', 'hexagon', 'hexagram', 'star-triangle-up', 'star-triangle-down', 'star-square', 'star-diamond', 'hourglass', 'bowtie']
        op_styles = {}
        for i, op in enumerate(ordered_ops):
            color = color_sequence[i % len(color_sequence)]
            symbol = marker_symbols[i % len(marker_symbols)]
            op_styles[op] = (color, symbol)
        import plotly.graph_objs as go
        fig = go.Figure()
        for op in ordered_ops:
            color, symbol = op_styles[op]
            op_data = pdf[pdf["operating_point"] == op]
            fig.add_trace(go.Scatter(
                x=op_data["start_time"],
                y=op_data["filcu_a"],
                mode='markers',
                name=op,
                marker=dict(color=color, symbol=symbol, size=8),
                showlegend=True
            ))
        fig.update_layout(
            title=f"Device SN: {device_id}, Tube SN: {tube_id}",
            xaxis_title="Timestamp",
            yaxis_title="Heater Current (A)",
            legend_title="Operating Point"
        )
        display(fig)

# Example usage: plot 10 healthy pairs (default)
plot_healthy_pairs()



## Correlation Analysis

 correlation matrix/heatmap between Heater Current, Focal Spot, High Voltage, EMission Current and Power (note: non-linear relations from tube characteristics)


In [0]:

import plotly.express as px

# Prepare relevant columns
corr_df = df_tube.select(
    F.col("filcu_a").alias("Heater Current (A)"),
    F.col("focal_spot").cast("string").alias("Focal Spot"),
    F.col("hv_kv_set").cast("double").alias("High Voltage (kV)"),
    F.col("tucu_ma_set").cast("double").alias("Emission Current (mA)"),
    (F.col("pwr_w_reached").cast("double")).alias("Power (W)"),
    F.col("grid_voltage").cast("double").alias("Grid Voltage (V)")
)

# Encode Focal Spot as numeric for correlation
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol="Focal Spot", outputCol="Focal Spot (index)")
corr_df_indexed = indexer.fit(corr_df).transform(corr_df)

# Select only numeric columns for correlation
numeric_cols = [
    "Heater Current (A)",
    "Focal Spot (index)",
    "High Voltage (kV)",
    "Emission Current (mA)",
    "Power (W)",
    "Grid Voltage (V)"
]
corr_pd = corr_df_indexed.select(numeric_cols).dropna().toPandas()

# Compute correlation matrix
corr_matrix = corr_pd.corr()

# Plot heatmap with defined size
fig = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale="RdBu",
    title="Correlation Matrix: Heater Current, Focal Spot, High Voltage, Emission Current, Power, Grid Voltage",
    width=1400*0.75,
    height=1200*0.75
)
display(fig)

